# Extension: Cross-Model Robustness of the Bias Audit

### Replicating the name-injection audit on a non-linear classifier

**Abdihafid Yusuf — C1010963**

---

## Purpose

The main audit was conducted on logistic regression. That choice was deliberate — for a linear model, SHAP's `LinearExplainer` returns *exact* Shapley values and each coefficient is directly interpretable as a feature's contribution, removing explainer approximation error as a source of uncertainty.

But it leaves an obvious question unanswered: **is the null result a property of the data, or an artefact of the model class?** Linear models cannot represent feature interactions. A name might carry no independent weight while still interacting with other CV features to shift decisions — a mechanism logistic regression is structurally unable to express.

This notebook replicates the entire Design B audit on a **Random Forest**, which captures non-linear interactions by construction. Three things change:

| | Logistic regression | Random Forest |
|---|---|---|
| Interactions | Cannot represent | Captured by construction |
| Explainer | `LinearExplainer` (exact) | `TreeExplainer` (exact for trees) |
| Direct attribution | Coefficients | Gini feature importances |

If the null holds under both, the finding is a property of the data and task rather than of one model family — a materially stronger claim, and one that closes limitation 4 of the main study.

**Run the main notebook first.** This notebook is self-contained and reloads the data, but the comparison in Section 6 is written against the main study's reported figures.

---
## 1. Setup and data preparation

The pipeline replicates the main study exactly: identical collision screening, identical name lists, identical injection procedure, identical audit sample size. Only the classifier changes.

In [ ]:
# %pip install pandas numpy scikit-learn matplotlib seaborn scipy shap --quiet

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Setup complete.')

In [ ]:
TRAIN_PATH = 'train.csv'
TEST_PATH  = 'test.csv'

df = pd.concat([pd.read_csv(TRAIN_PATH), pd.read_csv(TEST_PATH)], ignore_index=True)
df['fit'] = (df['label'] != 'No Fit').astype(int)

FIRST_NAMES = {
    'White_Female': ['Allison','Anne','Carrie','Emily','Jill','Laurie','Kristen','Meredith','Sarah'],
    'Black_Female': ['Aisha','Ebony','Keisha','Kenya','Latonya','Lakisha','Latoya','Tamika','Tanisha'],
    'White_Male':   ['Brad','Brendan','Geoffrey','Greg','Brett','Jay','Matthew','Neil'],
    'Black_Male':   ['Darnell','Hakim','Jermaine','Kareem','Jamal','Leroy','Rasheed','Tyrone'],
}
SURNAMES = {
    'White': ['Baker','Kelly','McCarthy','Murphy','Murray','OBrien','Ryan','Sullivan','Walsh'],
    'Black': ['Jackson','Jones','Robinson','Washington','Williams'],
}
PLACEBO_NAMES = ['Alex','Jordan','Taylor','Morgan','Casey','Riley','Quinn','Avery','Rowan','Sage']

GROUPS = list(FIRST_NAMES.keys())
REFERENCE = 'White_Male'
NAME_TOKENS = {n.lower() for g in GROUPS for n in FIRST_NAMES[g]}
ALL_NAME_TOKENS = (NAME_TOKENS
                   | {n.lower() for r in SURNAMES for n in SURNAMES[r]}
                   | {n.lower() for n in PLACEBO_NAMES})


def inject_name(resume_text, first, last):
    return f"{first} {last}\n{first.lower()}{last.lower()}@email.com\n{resume_text}"


def strip_names(text, first, last):
    t = re.sub(re.escape(first + last), 'XY', text, flags=re.IGNORECASE)
    t = re.sub(rf'\b{re.escape(first)}\b', 'X', t, flags=re.IGNORECASE)
    t = re.sub(rf'\b{re.escape(last)}\b',  'Y', t, flags=re.IGNORECASE)
    return t


# Screen out resumes already containing a candidate name token
collision_mask = df['resume_text'].apply(
    lambda t: bool(set(re.findall(r'[a-z]+', str(t).lower())) & ALL_NAME_TOKENS))
df_clean = df[~collision_mask].reset_index(drop=True)

print(f'Total pairs:        {len(df):,}')
print(f'Name collisions:    {collision_mask.sum():,} ({collision_mask.mean():.1%}) - excluded from audit')
print(f'Clean audit pool:   {len(df_clean):,}')

---
## 2. Balanced-injection training

Identical to Design B in the main study. Each training CV receives a name drawn at random from the four demographic groups **plus a placebo group** of demographically neutral names, assigned independently of the fit label.

The placebo group is included in training so that neutral names receive equal vocabulary exposure. Without this, the negative control in Section 5 has no valid noise floor to compare against.

### 2.1 Exposure matching — why sampling is uniform over names

The placebo control only works if demographic and neutral names receive **equal exposure** during training. Gini importance scales with how often a feature is encountered, so any imbalance in exposure translates directly into an apparent importance difference that has nothing to do with demographics.

Sampling by *group* creates exactly this imbalance. With four demographic groups and one placebo group each selected with probability 1/5, and unequal numbers of names within each, per-name exposure differs:

| Pool | Names | Per-name probability |
|---|---|---|
| White_Female / Black_Female | 9 each | 0.0222 |
| White_Male / Black_Male | 8 each | 0.0250 |
| Placebo | 10 | 0.0200 |

Demographic names would appear **1.18× more often** than placebo names on average. In testing, this alone produced an apparently significant placebo result (*p* = 0.024) that vanished once exposure was matched (*p* = 0.634).

The sampling below therefore draws uniformly from a flat roster of all names, so every name — demographic or placebo — has identical selection probability. This is a small implementation detail with a decisive effect on the validity of the negative control.

In [ ]:
train_B, audit_B = train_test_split(
    df_clean, test_size=0.35, random_state=RANDOM_STATE, stratify=df_clean['fit'])
train_B = pd.concat([train_B, df[collision_mask]], ignore_index=True).copy()

# Sampling is uniform over NAMES, not over groups. See section note below:
# sampling by group would give names in smaller groups higher per-name exposure,
# which inflates their Gini importance and invalidates the placebo comparison.
NAME_ROSTER = ([(n, g) for g in GROUPS for n in FIRST_NAMES[g]] +
               [(n, 'Placebo') for n in PLACEBO_NAMES])

assigned, injected = [], []
for _, row in train_B.iterrows():
    f, g = NAME_ROSTER[rng.integers(len(NAME_ROSTER))]
    race = rng.choice(['White','Black']) if g == 'Placebo' else g.split('_')[0]
    l = rng.choice(SURNAMES[race])
    assigned.append(g)
    injected.append(inject_name(row['resume_text'], f, l))

print(f'Name roster: {len(NAME_ROSTER)} names '
      f'({len(NAME_ROSTER)-len(PLACEBO_NAMES)} demographic + {len(PLACEBO_NAMES)} placebo)')
print(f'Per-name selection probability: {1/len(NAME_ROSTER):.5f} (identical for all names)\n')

train_B['assigned_group'] = assigned
train_B['inj_resume'] = injected

ct = pd.crosstab(train_B['assigned_group'], train_B['fit'])
chi2, p_chi, _, _ = stats.chi2_contingency(ct)
print('Advance rate by randomly assigned group:\n')
print(train_B.groupby('assigned_group')['fit'].agg(['mean','count']).round(4).to_string())
print(f'\nChi-square test of independence: chi2={chi2:.3f}, p={p_chi:.4f}')
print('Non-significance confirms names carry no information about the outcome.')

In [ ]:
vec_r = TfidfVectorizer(max_features=8000, stop_words='english', min_df=2)
vec_j = TfidfVectorizer(max_features=8000, stop_words='english', min_df=3)
R = vec_r.fit_transform(train_B['inj_resume'])
J = vec_j.fit_transform(train_B['job_description_text'].astype(str))

vec_sh = TfidfVectorizer(max_features=10000, stop_words='english', min_df=3)
vec_sh.fit(pd.concat([train_B['inj_resume'], train_B['job_description_text'].astype(str)]))

def overlap(a, b):
    A = vec_sh.transform(a); B = vec_sh.transform(b)
    num = np.asarray(A.multiply(B).sum(axis=1)).ravel()
    na = np.sqrt(np.asarray(A.multiply(A).sum(axis=1)).ravel())
    nb = np.sqrt(np.asarray(B.multiply(B).sum(axis=1)).ravel())
    return (num / (na*nb + 1e-9)).reshape(-1,1)

X_train = hstack([R, J, csr_matrix(
    overlap(train_B['inj_resume'], train_B['job_description_text'].astype(str)))]).tocsr()
y_train = train_B['fit'].values

feat_r = vec_r.get_feature_names_out()
feat_j = vec_j.get_feature_names_out()
feature_names = [f'R::{f}' for f in feat_r] + [f'J::{f}' for f in feat_j] + ['OVERLAP']

print(f'Feature space: {X_train.shape[1]:,}')
print(f'Name tokens in resume vocabulary: {len([f for f in feat_r if f in NAME_TOKENS])}/{len(NAME_TOKENS)}')

In [ ]:
t0 = time.time()
model_rf = RandomForestClassifier(
    n_estimators=150, max_depth=18, min_samples_leaf=2,
    n_jobs=-1, random_state=RANDOM_STATE)
model_rf.fit(X_train, y_train)
print(f'Random Forest trained in {time.time()-t0:.1f}s')
print(f'Training accuracy: {model_rf.score(X_train, y_train):.3f}')

# Held-out generalisation check on non-injected pairs
hold = audit_B.sample(min(1500, len(audit_B)), random_state=RANDOM_STATE+1)
X_hold = hstack([vec_r.transform(hold['resume_text'].astype(str)),
                 vec_j.transform(hold['job_description_text'].astype(str)),
                 csr_matrix(overlap(hold['resume_text'].astype(str),
                                    hold['job_description_text'].astype(str)))]).tocsr()
rf_test_acc = model_rf.score(X_hold, hold['fit'].values)
print(f'Held-out accuracy: {rf_test_acc:.3f}  (baseline {max(hold["fit"].mean(), 1-hold["fit"].mean()):.3f})')

---
## 3. Audit sample and paired analysis

Each held-out pair is duplicated across the four demographic conditions. As in the main study, variants are verified byte-identical after name removal before any inference is drawn.

In [ ]:
N_BASE = 400
base = audit_B.sample(min(N_BASE, len(audit_B)), random_state=RANDOM_STATE).reset_index(drop=True)
base['base_id'] = range(len(base))

records = []
for _, row in base.iterrows():
    for g in GROUPS:
        f = rng.choice(FIRST_NAMES[g])
        l = rng.choice(SURNAMES[g.split('_')[0]])
        records.append({
            'base_id': row['base_id'], 'group': g,
            'race': g.split('_')[0], 'gender': g.split('_')[1],
            'first': f, 'last': l, 'injected_name': f'{f} {l}',
            'resume': inject_name(row['resume_text'], f, l),
            'job_description': row['job_description_text'],
            'true_fit': row['fit'],
        })
A_df = pd.DataFrame(records)

# Verify the manipulation is clean
n_ok = 0
for bid, grp in A_df.groupby('base_id'):
    s = [strip_names(r['resume'], r['first'], r['last']) for _, r in grp.iterrows()]
    n_ok += (len(set(s)) == 1)
print(f'Variant sets identical after name removal: {n_ok}/{A_df["base_id"].nunique()}')
assert n_ok == A_df['base_id'].nunique(), 'Injection contaminated some resumes.'

X_audit = hstack([vec_r.transform(A_df['resume']),
                  vec_j.transform(A_df['job_description']),
                  csr_matrix(overlap(A_df['resume'], A_df['job_description']))]).tocsr()
A_df['proba'] = model_rf.predict_proba(X_audit)[:, 1]
A_df['pred']  = model_rf.predict(X_audit)

print(f'\nBase pairs: {len(base)} | variants: {len(A_df):,}\n')
print('Advance rate by group (%):')
print((A_df.groupby('group')['pred'].mean()*100).round(2).to_string())

In [ ]:
def paired_analysis(data, reference=REFERENCE, n_boot=2000):
    wide  = data.pivot(index='base_id', columns='group', values='proba')
    wpred = data.pivot(index='base_id', columns='group', values='pred')
    rows = []
    for g in [x for x in GROUPS if x != reference]:
        diff = (wide[g] - wide[reference]).dropna()
        t_stat, p_val = stats.ttest_rel(wide[g], wide[reference])
        d = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else 0.0
        boot = rng.choice(diff.values, size=(n_boot, len(diff)), replace=True).mean(axis=1)
        lo, hi = np.percentile(boot, [2.5, 97.5])
        flipped = int((wpred[g] != wpred[reference]).sum())
        rows.append({'group': g, 'mean_diff': diff.mean(), 'ci_low': lo, 'ci_high': hi,
                     'cohens_d': d, 't': t_stat, 'p_raw': p_val,
                     'decisions_flipped': flipped})
    out = pd.DataFrame(rows).sort_values('p_raw').reset_index(drop=True)
    k = len(out)
    out['p_holm'] = [min(1.0, (k-i)*p) for i, p in enumerate(out['p_raw'])]
    out['p_holm'] = out['p_holm'].cummax()
    out['significant'] = out['p_holm'] < 0.05
    return out, wide, wpred


def interpret_d(d):
    a = abs(d)
    return 'negligible' if a < 0.2 else 'small' if a < 0.5 else 'medium' if a < 0.8 else 'large'


paired_rf, wide_rf, wpred_rf = paired_analysis(A_df)
paired_rf['magnitude'] = paired_rf['cohens_d'].apply(interpret_d)

print(f'RANDOM FOREST — paired comparison against {REFERENCE}')
print(f'Matched sets: {len(wide_rf)}\n')
print(paired_rf[['group','mean_diff','ci_low','ci_high','cohens_d','magnitude',
                 'p_raw','p_holm','significant','decisions_flipped']].round(5).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
comps = [g for g in GROUPS if g != REFERENCE]

bp = axes[0].boxplot([(wide_rf[g]-wide_rf[REFERENCE]).values for g in comps],
                     tick_labels=comps, patch_artist=True, showmeans=True)
for p in bp['boxes']: p.set_facecolor('lightsteelblue')
axes[0].axhline(0, color='crimson', linestyle='--', linewidth=1.5)
axes[0].set_ylabel('Change in P(advance) vs identical White_Male CV')
axes[0].set_title('Random Forest: paired name effect')
axes[0].tick_params(axis='x', rotation=15)

o = paired_rf.set_index('group').reindex(comps)
axes[1].errorbar(o['mean_diff'], range(len(o)),
                 xerr=[o['mean_diff']-o['ci_low'], o['ci_high']-o['mean_diff']],
                 fmt='o', capsize=5, color='seagreen', markersize=8)
axes[1].axvline(0, color='crimson', linestyle='--')
axes[1].set_yticks(range(len(o))); axes[1].set_yticklabels(o.index)
axes[1].set_xlabel('Mean paired difference in P(advance)')
axes[1].set_title('Effect estimates with 95% bootstrap CIs')
plt.tight_layout(); plt.show()

---
## 4. Intersectional fairness metrics

In [ ]:
A_df['correct'] = (A_df['pred'] == A_df['true_fit']).astype(int)

fairness_rf = A_df.groupby('group').agg(
    selection_rate=('pred','mean'), mean_proba=('proba','mean'),
    accuracy=('correct','mean'), n=('pred','size'))
base_rate = fairness_rf.loc[REFERENCE, 'selection_rate']
fairness_rf['dpd'] = fairness_rf['selection_rate'] - base_rate
fairness_rf['di']  = fairness_rf['selection_rate'] / base_rate
fairness_rf = fairness_rf.sort_values('selection_rate', ascending=False)

disp = fairness_rf.copy()
disp['selection_%'] = (disp['selection_rate']*100).round(2)
disp['accuracy_%']  = (disp['accuracy']*100).round(2)
print('INTERSECTIONAL FAIRNESS METRICS (Random Forest)\n')
print(disp[['selection_%','dpd','di','accuracy_%','n']].round(4).to_string())
print()
print(f'Largest between-group gap: {fairness_rf["selection_rate"].max()-fairness_rf["selection_rate"].min():.4f}')
print(f'Minimum disparate impact ratio: {fairness_rf["di"].min():.3f} '
      f'({"BELOW" if fairness_rf["di"].min() < 0.80 else "above"} the 0.80 four-fifths threshold)')

by_race   = A_df.groupby('race')['pred'].mean()
by_gender = A_df.groupby('gender')['pred'].mean()
by_both   = A_df.groupby('group')['pred'].mean()
race_gap   = abs(by_race.diff().iloc[-1])
gender_gap = abs(by_gender.diff().iloc[-1])
inter_gap  = by_both.max() - by_both.min()
print()
print(f'Race-only gap:      {race_gap:.4f}')
print(f'Gender-only gap:    {gender_gap:.4f}')
print(f'Intersectional gap: {inter_gap:.4f}')

---
## 5. Direct attribution: Gini importance and placebo control

A Random Forest has no coefficients, so the analogue of the main study's coefficient audit is **Gini feature importance** — the total impurity reduction attributable to each feature across all trees.

The logic is unchanged. Because names were assigned at random and independently of the label, the expected importance of every name feature is approximately zero. Any systematic elevation represents signal the model manufactured from noise.

The placebo comparison remains the control that licenses the inference: if demographic names carry no more importance than neutral names given equal exposure, there is no evidence of demographic bias.

In [ ]:
imp_all = model_rf.feature_importances_
imp_r = imp_all[:len(feat_r)]

rows = []
for g in GROUPS:
    for n in FIRST_NAMES[g]:
        idx = np.where(feat_r == n.lower())[0]
        if len(idx):
            rows.append({'name': n, 'group': g, 'race': g.split('_')[0],
                         'gender': g.split('_')[1], 'importance': imp_r[idx[0]]})
name_imp = pd.DataFrame(rows)

placebo_rows = []
for n in PLACEBO_NAMES:
    idx = np.where(feat_r == n.lower())[0]
    if len(idx):
        placebo_rows.append({'name': n, 'importance': imp_r[idx[0]]})
placebo_imp = pd.DataFrame(placebo_rows)

print(f'Demographic names recovered: {len(name_imp)}/{len(NAME_TOKENS)}')
print(f'Placebo names recovered:     {len(placebo_imp)}/{len(PLACEBO_NAMES)}\n')
print('Gini importance by group:\n')
print(name_imp.groupby('group')['importance'].agg(['mean','std','count']).round(8).to_string())
print()
print(f'Name share of total resume-block importance: {name_imp["importance"].sum()/imp_r.sum():.4%}')

In [ ]:
print('SIGNIFICANCE TESTS ON NAME IMPORTANCE\n')

arrs = [name_imp[name_imp['group']==g]['importance'].values for g in GROUPS
        if (name_imp['group']==g).any()]
F, p_anova = stats.f_oneway(*arrs)
print(f'One-way ANOVA across four groups: F={F:.3f}, p={p_anova:.4f}')

w = name_imp[name_imp['race']=='White']['importance']
b = name_imp[name_imp['race']=='Black']['importance']
m_ = name_imp[name_imp['gender']=='Male']['importance']
f_ = name_imp[name_imp['gender']=='Female']['importance']

def hedges_g(x, y):
    nx, ny = len(x), len(y)
    sp = np.sqrt(((nx-1)*x.var(ddof=1) + (ny-1)*y.var(ddof=1)) / (nx+ny-2))
    d = (x.mean()-y.mean())/sp if sp > 0 else 0.0
    return d * (1 - 3/(4*(nx+ny) - 9))

tw, pw = stats.ttest_ind(w, b); tg, pg = stats.ttest_ind(m_, f_)
print(f'Race   (White vs Black):  t={tw:+.3f}, p={pw:.4f}, Hedges g={hedges_g(w,b):+.3f}')
print(f'Gender (Male vs Female):  t={tg:+.3f}, p={pg:.4f}, Hedges g={hedges_g(m_,f_):+.3f}')

print('\nPLACEBO CONTROL\n')
if len(placebo_imp) >= 4:
    tp, pp = stats.ttest_ind(name_imp['importance'], placebo_imp['importance'])
    print(f'Demographic names: mean={name_imp["importance"].mean():.8f}, n={len(name_imp)}')
    print(f'Placebo names:     mean={placebo_imp["importance"].mean():.8f}, n={len(placebo_imp)}')
    print(f'Difference: t={tp:+.3f}, p={pp:.4f}, Hedges g={hedges_g(name_imp["importance"], placebo_imp["importance"]):+.3f}')
    print()
    if pp < 0.05 and name_imp['importance'].mean() > placebo_imp['importance'].mean():
        print('Demographic names carry significantly MORE importance than neutral names')
        print('given equal exposure. This exceeds the noise floor.')
    else:
        print('Demographic names do not carry significantly more importance than neutral')
        print('names given equal exposure. Consistent with the estimation noise floor.')
else:
    print(f'Only {len(placebo_imp)} placebo names recovered - too few for comparison.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

o = name_imp.sort_values('importance')
cols = ['#d73027' if 'Black' in g else '#4575b4' for g in o['group']]
axes[0].barh(range(len(o)), o['importance'], color=cols)
axes[0].set_yticks(range(len(o)))
axes[0].set_yticklabels([f"{r['name']} ({r['group'].replace('_',' ')})" for _, r in o.iterrows()], fontsize=7)
axes[0].set_xlabel('Gini importance')
axes[0].set_title('Random Forest importance of each demographic name\n(names randomised, so weight is spurious)')
axes[0].legend(handles=[mpatches.Patch(color='#4575b4', label='White'),
                        mpatches.Patch(color='#d73027', label='Black')], fontsize=8)

grp = name_imp.groupby('group')['importance'].agg(['mean','std','count'])
grp['se'] = grp['std']/np.sqrt(grp['count']); grp = grp.reindex(GROUPS)
cols2 = ['#d73027' if 'Black' in g else '#4575b4' for g in grp.index]
axes[1].bar(grp.index, grp['mean'], yerr=grp['se'], capsize=5, color=cols2)
if len(placebo_imp) >= 4:
    axes[1].axhline(placebo_imp['importance'].mean(), color='black', linestyle='--',
                    linewidth=1.5, label='Placebo mean (noise floor)')
    axes[1].legend()
axes[1].set_ylabel('Mean Gini importance')
axes[1].set_title('Mean name importance by group')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

---
## 6. SHAP analysis with TreeExplainer

`TreeExplainer` computes exact Shapley values for tree ensembles. It is considerably more expensive than `LinearExplainer`, so a reduced sample is used.

The cell below is defensive by design: it handles the differing output shapes returned across SHAP versions for binary classifiers (some return a list of two arrays, others a three-dimensional array), and degrades gracefully if computation fails. Gini importance in Section 5 already provides an exact global attribution measure, so the analysis does not depend on this section succeeding.

In [ ]:
SHAP_N = 200   # reduce further if this is slow on your machine

shap_ok = False
try:
    import shap
    shap_idx = rng.choice(len(A_df), size=min(SHAP_N, len(A_df)), replace=False)
    A_shap = A_df.iloc[shap_idx].reset_index(drop=True)

    X_shap = hstack([vec_r.transform(A_shap['resume']),
                     vec_j.transform(A_shap['job_description']),
                     csr_matrix(overlap(A_shap['resume'], A_shap['job_description']))]).tocsr()

    print(f'Computing TreeExplainer values for {X_shap.shape[0]} variants...')
    t0 = time.time()
    explainer = shap.TreeExplainer(model_rf)
    raw = explainer.shap_values(X_shap.toarray(), check_additivity=False)

    # Normalise across SHAP versions: want a 2D array for the positive class
    if isinstance(raw, list):
        sv = raw[1]
    elif isinstance(raw, np.ndarray) and raw.ndim == 3:
        sv = raw[:, :, 1]
    else:
        sv = raw
    sv = np.asarray(sv)

    print(f'Done in {time.time()-t0:.1f}s | shape {sv.shape}')
    shap_ok = True
except Exception as e:
    print(f'TreeExplainer did not complete: {type(e).__name__}: {e}')
    print('Proceeding using Gini importance from Section 5 as the attribution measure.')

In [ ]:
if shap_ok:
    mean_abs = np.abs(sv).mean(axis=0)
    top = np.argsort(mean_abs)[-20:]
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh([feature_names[i] for i in top], mean_abs[top], color='seagreen')
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('Random Forest: global feature importance (TreeExplainer)')
    plt.tight_layout(); plt.show()

    name_feat_idx = [i for i, f in enumerate(feature_names)
                     if f.startswith('R::') and f[3:] in NAME_TOKENS]
    if name_feat_idx:
        share = np.abs(sv[:, name_feat_idx]).sum() / np.abs(sv).sum()
        print(f'Share of total SHAP attribution on name tokens: {share:.4%}')

        group_shap_rf = {}
        for g in GROUPS:
            mask = (A_shap['group'] == g).values
            if mask.sum():
                group_shap_rf[g] = float(np.abs(sv[mask][:, name_feat_idx]).mean())
        print('\nMean |SHAP| on name features by group:')
        for g, v in sorted(group_shap_rf.items(), key=lambda kv: -kv[1]):
            print(f'  {g:14s} {v:.8f}')
else:
    print('SHAP section skipped. Section 5 (Gini importance) provides exact global attribution.')

---
## 7. Cross-model comparison

The central question of this extension: does the conclusion depend on the model class?

In [ ]:
print('=' * 74)
print('CROSS-MODEL COMPARISON — DOES THE NULL HOLD?')
print('=' * 74)
print()
print('RANDOM FOREST (this notebook)')
print(f'  Held-out accuracy        {rf_test_acc:.3f}')
print(f'  Name tokens in vocab     {len(name_imp)}/{len(NAME_TOKENS)}')
print(f'  Names independent        chi2 p={p_chi:.4f}')
print()
print('  Paired effects vs identical White_Male CV:')
for _, r in paired_rf.iterrows():
    mark = '*' if r['significant'] else ' '
    print(f"    {r['group']:14s} d(P)={r['mean_diff']:+.5f}  Cohen's d={r['cohens_d']:+.3f} "
          f"({interpret_d(r['cohens_d']):10s})  p_holm={r['p_holm']:.4f}{mark}  flipped={r['decisions_flipped']}")
print('    * significant after Holm-Bonferroni correction')
print()
print(f'  Significant comparisons  {int(paired_rf["significant"].sum())}/3')
print(f'  Largest |Cohen\'s d|      {paired_rf["cohens_d"].abs().max():.3f}')
print(f'  Max decisions flipped    {paired_rf["decisions_flipped"].max()}/{len(wide_rf)} '
      f'({paired_rf["decisions_flipped"].max()/len(wide_rf):.1%})')
print()
print('  Attribution audit:')
print(f'    ANOVA across groups    F={F:.3f}, p={p_anova:.4f}')
print(f'    Race effect            p={pw:.4f}, Hedges g={hedges_g(w,b):+.3f}')
print(f'    Gender effect          p={pg:.4f}, Hedges g={hedges_g(m_,f_):+.3f}')
if len(placebo_imp) >= 4:
    print(f'    vs placebo control     p={pp:.4f}')
print()
print('  Fairness:')
print(f'    Largest selection gap  {inter_gap:.4f}')
print(f'    Min disparate impact   {fairness_rf["di"].min():.3f} (four-fifths threshold: 0.80)')
print()
print('-' * 74)
print('COMPARE AGAINST THE MAIN STUDY (logistic regression):')
print('  Enter the figures from the main notebook summary to complete this table')
print('  in the write-up. The key comparison is whether BOTH models show:')
print('    (a) negligible effect sizes regardless of statistical significance,')
print('    (b) null attribution audits, and')
print('    (c) name importance within the placebo noise floor.')
print('=' * 74)

In [ ]:
paired_rf.to_csv('rf_paired_analysis.csv', index=False)
fairness_rf.to_csv('rf_fairness_metrics.csv')
name_imp.to_csv('rf_name_importance.csv', index=False)
if len(placebo_imp):
    placebo_imp.to_csv('rf_placebo_importance.csv', index=False)

print('Exported:')
for f in ['rf_paired_analysis.csv','rf_fairness_metrics.csv','rf_name_importance.csv']:
    print(f'  {f}')

---
## 8. Interpretation

### What this extension establishes

The main study audited a linear model. Linear models cannot represent interactions between features, so a sceptical reader could object that a name might influence decisions *through interaction* with other CV content — a mechanism logistic regression is structurally unable to capture, and therefore unable to detect.

A Random Forest captures interactions by construction. Replicating the audit on it tests that objection directly.

**If both models show negligible effects, null attribution audits, and name importance within the placebo noise floor**, the conclusion is a property of the data and task rather than an artefact of model class. This is a substantially stronger claim than the main study alone supports, and it closes limitation 4.

**If the Random Forest reveals effects the linear model missed**, that is a more interesting finding still: it would show that interaction effects are the mechanism through which name bias operates, and that linear-model audits systematically under-detect it. Either outcome is publishable in the write-up.

### A methodological point worth making explicitly

The two models require different explainers — `LinearExplainer` and `TreeExplainer` — and different direct attribution measures — coefficients and Gini importance. That the same audit protocol transfers across both, with the same injection procedure, the same matched-set inference, and the same placebo control, demonstrates the protocol is **explainer-agnostic**. That generality is itself a contribution: it is the audit design, not the choice of explainer, that carries the inferential weight.

### The remaining boundary

Both models use bag-of-words representations, in which a name is an isolated token carrying no context. Neither can represent the contextual name effects that transformer-based screening systems might exhibit — where a name interacts semantically with surrounding content through attention.

The Design A finding from the main study, that bag-of-words models are structurally incapable of direct name discrimination, combined with the null established here across two model families, points to a clear conclusion for the discussion chapter: **the fairness risk in automated CV screening lies with contextual, embedding-based models rather than the bag-of-words pipelines examined here.** This study delineates where the risk is not, and identifies where future work should look.

---

## References

Bertrand, M. & Mullainathan, S. (2004) 'Are Emily and Greg more employable than Lakisha and Jamal? A field experiment on labor market discrimination', *American Economic Review*, 94(4), pp. 991–1013.

Breiman, L. (2001) 'Random forests', *Machine Learning*, 45(1), pp. 5–32.

Lundberg, S.M., Erion, G., Chen, H., DeGrave, A., Prutkin, J.M., Nair, B., Katz, R., Himmelfarb, J., Bansal, N. & Lee, S.I. (2020) 'From local explanations to global understanding with explainable AI for trees', *Nature Machine Intelligence*, 2(1), pp. 56–67.

Slack, D., Hilgard, S., Jia, E., Singh, S. & Lakkaraju, H. (2020) 'Fooling LIME and SHAP: adversarial attacks on post hoc explanation methods', *Proceedings of the AAAI/ACM Conference on AI, Ethics, and Society*, pp. 180–186.